In [7]:
import pandas as pd
import os

In [8]:
base_dir = os.getcwd()

# 1) Extraction ( convert different files to clean DataFrame)

In [9]:
ord_df = pd.read_csv(os.path.join(base_dir,"Data","orders.csv"))

cust_df = pd.read_json(os.path.join(base_dir,"Data","customers.json"))

exrates_df = pd.read_json(os.path.join(base_dir,"Data","exchange_rates.json"))

product_df = pd.read_csv(os.path.join(base_dir,"Data","products.csv"),sep="|")

returns_df = pd.read_csv(os.path.join(base_dir,"Data","returns.tsv"),sep="\t")

web_df = pd.read_csv(os.path.join(base_dir,"Data","web_events.log")
                     ,sep='|'
                     ,header=None
                     )

In [10]:
web_df.columns=[
        "timestamp",
        "session",
        "customer_id",
        "event",
        "product_id"
    ]

def clean_rows(value):
    return value.strip().split("=")[1]

columns =["session","customer_id","event","product_id"]

for col in columns:
    web_df[col] = web_df[col].apply(clean_rows)

In [11]:
ord_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 919 entries, 0 to 918
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     919 non-null    object 
 1   customer_id  866 non-null    object 
 2   product_id   918 non-null    object 
 3   quantity     919 non-null    int64  
 4   amount       886 non-null    float64
 5   currency     919 non-null    object 
 6   order_date   918 non-null    object 
dtypes: float64(1), int64(1), object(5)
memory usage: 50.4+ KB


## country_df -> Flatenning the customers address

### Method-1 : for loop

In [12]:
cust_df = pd.read_json(os.path.join(base_dir,"Data","customers.json"))

city=[]
country=[]

for address in cust_df["address"]:
    city.append(address["city"])
    country.append(address["country"])
cust_df["city"] = city
cust_df["country"] = country

cust_df = cust_df.drop(columns = ["address"])

cust_df

,customer_id,name,email,signup_date,is_premium,city,country
0,C0001,Customer 1,user1@example.com,2025-12-07,True,London,India
1,C0002,Customer 2,user2@example.com,2025-09-19,False,Mumbai,U.S.A
2,C0003,Customer 3,None,2025-09-28,True,Berlin,India
3,C0004,Customer 4,user4@example.com,2024-03-07,True,Mumbai,United Kingdom
4,C0005,Customer 5,user5@example.com,2025-03-23,False,Chennai,United States
...,...,...,...,...,...,...,...
115,C0116,Customer 116,user116@example.com,2024-12-09,False,London,United Kingdom
116,C0117,Customer 117,user117@example.com,2024-09-07,True,Chennai,DE
117,C0118,Customer 118,user118@example.com,2025-01-08,True,Berlin,U.S.A
118,C0119,Customer 119,user119@example.com,2025-04-23,True,Berlin,USA


###  Method-2 : using .apply()

In [13]:
cust_df = pd.read_json(os.path.join(base_dir,"Data","customers.json"))

cust_df["city"] = cust_df["address"].apply(lambda x : x["city"])

cust_df["country"] = cust_df["address"].apply(lambda x : x["country"])


cust_df

,customer_id,name,email,signup_date,address,is_premium,city,country
0,C0001,Customer 1,user1@example.com,2025-12-07,"{'city': 'London', 'country': 'India'}",True,London,India
1,C0002,Customer 2,user2@example.com,2025-09-19,"{'city': 'Mumbai', 'country': 'U.S.A'}",False,Mumbai,U.S.A
2,C0003,Customer 3,None,2025-09-28,"{'city': 'Berlin', 'country': 'India'}",True,Berlin,India
3,C0004,Customer 4,user4@example.com,2024-03-07,"{'city': 'Mumbai', 'country': 'United Kingdom'}",True,Mumbai,United Kingdom
4,C0005,Customer 5,user5@example.com,2025-03-23,"{'city': 'Chennai', 'country': 'United States'}",False,Chennai,United States
...,...,...,...,...,...,...,...,...
115,C0116,Customer 116,user116@example.com,2024-12-09,"{'city': 'London', 'country': 'United Kingdom'}",False,London,United Kingdom
116,C0117,Customer 117,user117@example.com,2024-09-07,"{'city': 'Chennai', 'country': 'DE'}",True,Chennai,DE
117,C0118,Customer 118,user118@example.com,2025-01-08,"{'city': 'Berlin', 'country': 'U.S.A'}",True,Berlin,U.S.A
118,C0119,Customer 119,user119@example.com,2025-04-23,"{'city': 'Berlin', 'country': 'USA'}",True,Berlin,USA


In [14]:
cust_df["address"].to_list()

[{'city': 'London', 'country': 'India'},
 {'city': 'Mumbai', 'country': 'U.S.A'},
 {'city': 'Berlin', 'country': 'India'},
 {'city': 'Mumbai', 'country': 'United Kingdom'},
 {'city': 'Chennai', 'country': 'United States'},
 {'city': 'Berlin', 'country': 'UK'},
 {'city': 'London', 'country': 'UK'},
 {'city': 'Mumbai', 'country': 'U.S.A'},
 {'city': 'Mumbai', 'country': 'UK'},
 {'city': 'Berlin', 'country': 'IN'},
 {'city': 'New York', 'country': 'IN'},
 {'city': 'Berlin', 'country': 'UK'},
 {'city': 'Chennai', 'country': 'United States'},
 {'city': 'London', 'country': 'DE'},
 {'city': 'Berlin', 'country': 'USA'},
 {'city': 'Berlin', 'country': 'UK'},
 {'city': 'New York', 'country': 'United States'},
 {'city': 'Mumbai', 'country': 'IN'},
 {'city': 'London', 'country': 'India'},
 {'city': 'Chennai', 'country': 'U.S.A'},
 {'city': 'New York', 'country': 'U.S.A'},
 {'city': 'Mumbai', 'country': 'IN'},
 {'city': 'Mumbai', 'country': 'IN'},
 {'city': 'Berlin', 'country': 'Germany'},
 {'city

###  Method-3 : optimized approach

In [15]:
cust_df = pd.read_json(os.path.join(base_dir,"Data","customers.json"))

cust_df[["city","country"]] = pd.DataFrame(
    cust_df["address"].to_list() # list of dictionary
)

cust_df = cust_df.drop(columns = ["address"])

cust_df

,customer_id,name,email,signup_date,is_premium,city,country
0,C0001,Customer 1,user1@example.com,2025-12-07,True,London,India
1,C0002,Customer 2,user2@example.com,2025-09-19,False,Mumbai,U.S.A
2,C0003,Customer 3,None,2025-09-28,True,Berlin,India
3,C0004,Customer 4,user4@example.com,2024-03-07,True,Mumbai,United Kingdom
4,C0005,Customer 5,user5@example.com,2025-03-23,False,Chennai,United States
...,...,...,...,...,...,...,...
115,C0116,Customer 116,user116@example.com,2024-12-09,False,London,United Kingdom
116,C0117,Customer 117,user117@example.com,2024-09-07,True,Chennai,DE
117,C0118,Customer 118,user118@example.com,2025-01-08,True,Berlin,U.S.A
118,C0119,Customer 119,user119@example.com,2025-04-23,True,Berlin,USA


### Reordering the columns

In [16]:
# cust_df = cust_df[
#     [
#         "customer_id",
#         "name",
#         "email",
#         "signup_date",
#         "city",
#         "country",
#         "is_premium"
#     ]
# ]
# cust_df

# 2) Transformation

## 1.a) Trim leading/trailing whitespace of string fields

In [17]:
# strip cleaning for only one DataFrame

ord_columns = ord_df.select_dtypes(include="object").columns

ord_columns
for column in ord_columns:
    ord_df[column] = ord_df[column].str.strip()
    
ord_df

,order_id,customer_id,product_id,quantity,amount,currency,order_date
0,O00545,C0115,P0011,3,33.78,INR,2026-04-28
1,O00071,C0048,P0037,4,399.24,EUR,2026-03-21
2,O00550,C0103,P0059,4,494.92,EUR,2026-03-16
3,O00528,C0081,P0018,2,286.14,USD,2026-02-04
4,O00456,C0034,P0040,3,352.80,INR,20/04/2026
...,...,...,...,...,...,...,...
914,O00360,C0037,P0051,5,235.80,INR,02-21-2026
915,O00128,C0082,P0099,3,1129.47,EUR,2026-06-30
916,O00481,C0069,P0040,5,588.00,JPY,07/02/2026
917,O00877,C0090,P0022,4,1599.72,INR,2026*04*11


In [18]:
data  = [
        ord_df,
        cust_df,
        exrates_df,
        product_df,
        web_df
        ]

### -> Debug: Check if any DataFrame is marked as a potential copy (view).

In [19]:
for i, df in enumerate(data):
    print(i, df._is_copy)

0 None
1 None
2 None
3 None
4 None


### -> Triming , Tailing and leading space generic function for every DataFrame

In [20]:

def clean_strip(df):
    str_columns = df.select_dtypes(include = "object").columns

    for column in str_columns:
        df[column] = df[column].str.strip()

for df in data:
    clean_strip(df)
  

## 1.b) Standardise_case

In [21]:
upper_columns = {
            "order_id",
            "customer_id",
            "product_id",
            "currency",
            "return_id"
            }

title_columns = {
            "product_name",
            "category",
            "name",
            "city"
            }     

lower_columns ={
            "email",
            "event",
            "country",

            }

capitalize_columns = {
            "reason"
            }


### -> Standardize case generic function for every DataFrame

In [22]:

for df in data:

    str_columns = df.select_dtypes(include = ["object"]).columns

    for column in str_columns:

        if column in upper_columns:
            df[column] = df[column].str.upper()

        elif column in title_columns:
            df[column] = df[column].str.title()
        
        elif column in lower_columns:
            df[column] = df[column].str.lower()

        elif column in capitalize_columns:
            df[column] = df[column].str.capitalize()
df

,timestamp,session,customer_id,event,product_id
0,2026-05-26T18:53:34,s5261,C0021,checkout,P0030
1,2026-03-02T04:48:04,s1248,C0036,remove_from_cart,P0006
2,2026-03-05T19:45:46,s5846,C0044,purchase,P0026
3,2026-01-10T22:24:41,s5493,C0047,remove_from_cart,P0043
4,2026-03-06T23:46:34,s1202,C0016,remove_from_cart,P0037
...,...,...,...,...,...
3995,2026-02-11T04:50:53,s1786,C0042,purchase,P0054
3996,2026-05-21T07:01:50,s8471,C0083,checkout,P0041
3997,2026-03-19T08:33:23,s6731,C0118,remove_from_cart,P0048
3998,2026-01-15T11:08:36,s9141,C0052,add_to_cart,P0022


## 2) Must be unique after trimmingDrop(drop duplicates, keep first) of order_id 

### -> itterrows() = how it works ?

In [23]:
# iterrows()

# for index , row in ord_df.iterrows():
#     print(index)
#     print(row["order_id"])
# itertuples()

### -> Finding the length of before and after duplicates 

In [24]:

seen=set()
cnt=0

print("Length before removing duplicates :",len(ord_df["order_id"]))

for row in ord_df.itertuples():
    if row.order_id in seen:
        print(row)
    seen.add(row.order_id)

print("Length after removig duplicates :",len(seen))

 

Length before removing duplicates : 919
Pandas(Index=147, order_id='O00826', customer_id='C0001', product_id='P0043', quantity=3, amount=1050.66, currency='INR', order_date='16/04/2026')
Pandas(Index=153, order_id='O00516', customer_id='C0009', product_id='P0007', quantity=1, amount=55.44, currency='EUR', order_date='04-27-2026')
Pandas(Index=206, order_id='O00229', customer_id='C0057', product_id='P0006', quantity=3, amount=894.27, currency='USD', order_date='01-17-2026')
Pandas(Index=286, order_id='O00684', customer_id='C0060', product_id='P0005', quantity=-2, amount=0.0, currency='GBP', order_date='30/01/2026')
Pandas(Index=460, order_id='O00431', customer_id='C0114', product_id='P0002', quantity=5, amount=1950.05, currency='USD', order_date='16/01/2026')
Pandas(Index=528, order_id='O00796', customer_id='C0014', product_id='P0055', quantity=2, amount=312.32, currency='GBP', order_date='04-03-2026')
Pandas(Index=562, order_id='O00455', customer_id='C0081', product_id='P0006', quantit

In [25]:

duplicates  = ord_df[ord_df["order_id"].duplicated(keep='first')]
duplicates["reason"] = "Duplicate order_id"
duplicates


/tmp/ipykernel_472904/1147269761.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  duplicates["reason"] = "Duplicate order_id"


,order_id,customer_id,product_id,quantity,amount,currency,order_date,reason
147,O00826,C0001,P0043,3,1050.66,INR,16/04/2026,Duplicate order_id
153,O00516,C0009,P0007,1,55.44,EUR,04-27-2026,Duplicate order_id
206,O00229,C0057,P0006,3,894.27,USD,01-17-2026,Duplicate order_id
286,O00684,C0060,P0005,-2,0.00,GBP,30/01/2026,Duplicate order_id
460,O00431,C0114,P0002,5,1950.05,USD,16/01/2026,Duplicate order_id
528,O00796,C0014,P0055,2,312.32,GBP,04-03-2026,Duplicate order_id
562,O00455,C0081,P0006,4,1192.36,USD,01-15-2026,Duplicate order_id
569,O00466,C0040,P0010,1,39.13,INR,05-04-2026,Duplicate order_id
600,O00217,C0087,P0012,2,114.48,GBP,05-27-2026,Duplicate order_id
606,O00391,C0080,P0021,1,NaN,INR,06-08-2026,Duplicate order_id


In [26]:
len(duplicates)

18

In [27]:
ord_df.drop_duplicates(subset="order_id",keep="first",inplace=True)
ord_df

,order_id,customer_id,product_id,quantity,amount,currency,order_date
0,O00545,C0115,P0011,3,33.78,INR,2026-04-28
1,O00071,C0048,P0037,4,399.24,EUR,2026-03-21
2,O00550,C0103,P0059,4,494.92,EUR,2026-03-16
3,O00528,C0081,P0018,2,286.14,USD,2026-02-04
4,O00456,C0034,P0040,3,352.80,INR,20/04/2026
...,...,...,...,...,...,...,...
914,O00360,C0037,P0051,5,235.80,INR,02-21-2026
915,O00128,C0082,P0099,3,1129.47,EUR,2026-06-30
916,O00481,C0069,P0040,5,588.00,JPY,07/02/2026
917,O00877,C0090,P0022,4,1599.72,INR,2026*04*11


## 3) Validate order_id ,product_id,order_date Must not be null

In [28]:
validate_col = ["order_id","product_id","order_date"]

null_rows = ord_df[ord_df[validate_col].isnull().any(axis=1)]
ord_df[validate_col].isnull().any(axis=0)

# .any()          -> Checks if any value is True in each column (axis=0)
# .any(axis=1)    -> Checks if any value is True in each row (axis=1)

order_id      False
product_id     True
order_date     True
dtype: bool

In [29]:
ord_df = ord_df.drop(index=null_rows.index)

In [30]:
# .all()          -> Checks if all values are True in each column (axis=0)
# .all(axis=1)    -> Checks if all values are True in each row (axis=1)

## 4) customer_id must be present and exist in customers

In [31]:
cust_df

,customer_id,name,email,signup_date,is_premium,city,country
0,C0001,Customer 1,user1@example.com,2025-12-07,True,London,india
1,C0002,Customer 2,user2@example.com,2025-09-19,False,Mumbai,u.s.a
2,C0003,Customer 3,None,2025-09-28,True,Berlin,india
3,C0004,Customer 4,user4@example.com,2024-03-07,True,Mumbai,united kingdom
4,C0005,Customer 5,user5@example.com,2025-03-23,False,Chennai,united states
...,...,...,...,...,...,...,...
115,C0116,Customer 116,user116@example.com,2024-12-09,False,London,united kingdom
116,C0117,Customer 117,user117@example.com,2024-09-07,True,Chennai,de
117,C0118,Customer 118,user118@example.com,2025-01-08,True,Berlin,u.s.a
118,C0119,Customer 119,user119@example.com,2025-04-23,True,Berlin,usa


In [32]:
ord_df

,order_id,customer_id,product_id,quantity,amount,currency,order_date
0,O00545,C0115,P0011,3,33.78,INR,2026-04-28
1,O00071,C0048,P0037,4,399.24,EUR,2026-03-21
2,O00550,C0103,P0059,4,494.92,EUR,2026-03-16
3,O00528,C0081,P0018,2,286.14,USD,2026-02-04
4,O00456,C0034,P0040,3,352.80,INR,20/04/2026
...,...,...,...,...,...,...,...
913,O00813,C0113,P0053,4,1555.88,EUR,02-01-2026
914,O00360,C0037,P0051,5,235.80,INR,02-21-2026
915,O00128,C0082,P0099,3,1129.47,EUR,2026-06-30
916,O00481,C0069,P0040,5,588.00,JPY,07/02/2026


In [33]:
lst_validate_customers = cust_df["customer_id"]
set_validate_customers = set(lst_validate_customers)


In [34]:
rejected_indexes=[]
for ind , row in ord_df.iterrows():
    if row["customer_id"] not in (set_validate_customers):
        rejected_indexes.append(ind)   
ord_df.drop(index=rejected_indexes,inplace=True)

In [35]:
ord_df.shape[0]

847

 ## 5) validate quantity Must be numeric and ≥ 0 (reject row)

In [36]:
invalid_quantity = ord_df[ord_df["quantity"]<0]
print(invalid_quantity)
print(len(invalid_quantity))
ord_df = ord_df.drop(index=invalid_quantity.index)

    order_id customer_id product_id  quantity  amount currency  order_date
19    O00093       C0026      P0016        -2     0.0      GBP  2026-04-03
118   O00329       C0112      P0008        -2     0.0      USD  06-15-2026
122   O00369       C0087      P0032        -2     0.0      INR  2026-01-28
164   O00047       C0119      P0045        -1     0.0      INR  06-14-2026
181   O00324       C0083      P0005        -2     0.0      GBP  29/06/2026
193   O00684       C0060      P0005        -2     0.0      GBP  30/01/2026
230   O00701       C0043      P0041        -1     0.0      INR  2026-01-20
252   O00812       C0019      P0015        -1     0.0      INR  03/04/2026
265   O00605       C0047      P0024        -2     0.0      USD  2026-06-24
282   O00498       C0061      P0034        -1     0.0      INR  08/04/2026
293   O00515       C0006      P0042        -2     0.0      EUR  01-05-2026
311   O00180       C0024      P0028        -2     0.0      GBP  2026-03-01
351   O00716       C0003 

## 6) Validate price Must be numeric and ≥ 0 (not blank)

In [37]:
len(ord_df)

824

In [38]:
invalid_amount = ord_df[(ord_df["amount"] < 0) | (ord_df["amount"].isnull())] 
invalid_amount

,order_id,customer_id,product_id,quantity,amount,currency,order_date
9,O00564,C0013,P0023,2,-1.0,INR,25/06/2026
39,O00658,C0115,P0055,5,NaN,GBP,20/04/2026
139,O00310,C0096,P0035,3,NaN,USD,06-20-2026
168,O00779,C0048,P0044,4,NaN,GBP,2026-05-16
195,O00164,C0012,P0029,5,NaN,INR,2026-04-03
208,O00037,C0015,P0002,0,NaN,USD,2026-02-03
225,O00391,C0080,P0021,1,NaN,INR,06-08-2026
229,O00485,C0020,P0056,2,NaN,USD,09/04/2026
243,O00074,C0051,P0016,4,NaN,GBP,2026-03-31
316,O00646,C0076,P0060,4,NaN,USD,05-05-2026


In [39]:
ord_df = ord_df.drop(index=invalid_amount.index)

In [40]:
len(ord_df)

797

## 7) Parse to a real date from any of 3 formats (YYYY-MM-DD, DD/MM/YYYY, MM-DD-YYYY) 

In [41]:
mixed_format = pd.to_datetime(ord_df["order_date"],errors = "coerce",format="mixed")

In [42]:
mixed_format

0     2026-04-28
1     2026-03-21
2     2026-03-16
3     2026-02-04
4     2026-04-20
         ...    
913   2026-02-01
914   2026-02-21
915   2026-06-30
916   2026-07-02
917          NaT
Name: order_date, Length: 797, dtype: datetime64[ns]

In [43]:
import datetime as dt

formats = [
        "%Y-%m-%d",
        "%d/%m/%Y",
        "%m-%d-%Y"
    ]

rejected_indexes=[]

for index , row in ord_df.iterrows():
    date = str(row["order_date"]).strip()
    valid = False

    for fmt in formats:
        try:
            parsed_date = dt.datetime.strptime(date,fmt)
            ord_df.at[index, "order_date"] = parsed_date
            valid = True
            break
            
        except ValueError:
            continue
   
    if not valid:
        rejected_indexes.append(index)

rejected_date_rows = ord_df.loc[rejected_indexes]

ord_df = ord_df.drop(index=rejected_indexes)

ord_df["order_date"] = pd.to_datetime(ord_df["order_date"])
    
rejected_date_rows


,order_id,customer_id,product_id,quantity,amount,currency,order_date
917,O00877,C0090,P0022,4,1599.72,INR,2026*04*11


In [44]:
ord_df["order_date"].sort_values().head(1)

454   2026-01-01
Name: order_date, dtype: datetime64[ns]

In [45]:
ord_df["order_date"].sort_values().tail(1)

734   2026-06-30
Name: order_date, dtype: datetime64[ns]

## 8 ) Standardise country Map variants to canonical (USA / India / UK / Germany)

In [46]:
cust_df.columns

Index(['customer_id', 'name', 'email', 'signup_date', 'is_premium', 'city',
       'country'],
      dtype='object')

In [47]:
cust_df

,customer_id,name,email,signup_date,is_premium,city,country
0,C0001,Customer 1,user1@example.com,2025-12-07,True,London,india
1,C0002,Customer 2,user2@example.com,2025-09-19,False,Mumbai,u.s.a
2,C0003,Customer 3,None,2025-09-28,True,Berlin,india
3,C0004,Customer 4,user4@example.com,2024-03-07,True,Mumbai,united kingdom
4,C0005,Customer 5,user5@example.com,2025-03-23,False,Chennai,united states
...,...,...,...,...,...,...,...
115,C0116,Customer 116,user116@example.com,2024-12-09,False,London,united kingdom
116,C0117,Customer 117,user117@example.com,2024-09-07,True,Chennai,de
117,C0118,Customer 118,user118@example.com,2025-01-08,True,Berlin,u.s.a
118,C0119,Customer 119,user119@example.com,2025-04-23,True,Berlin,usa


In [48]:
country_map ={
    "uk" : "UK",
    "united kingdom" : "UK",
    "united states" : "USA",
    "usa" : "USA" ,
    "u.s.a" : "USA" ,
    "de" : "Germany" ,
    "germany" : "Germany",
    "india" : "India",
    "in" : "India",  
}

In [49]:
cust_df["country"] = cust_df["country"].apply(lambda x : country_map[x] )



In [50]:
unique = set(cust_df["country"])
unique

{'Germany', 'India', 'UK', 'USA'}

## 9 ) Add email_present boolean

In [51]:
cust_df.columns

Index(['customer_id', 'name', 'email', 'signup_date', 'is_premium', 'city',
       'country'],
      dtype='object')

In [52]:
cust_df["email_present"] = ~cust_df["email"].isnull()

In [53]:
cust_df

,customer_id,name,email,signup_date,is_premium,city,country,email_present
0,C0001,Customer 1,user1@example.com,2025-12-07,True,London,India,True
1,C0002,Customer 2,user2@example.com,2025-09-19,False,Mumbai,USA,True
2,C0003,Customer 3,None,2025-09-28,True,Berlin,India,False
3,C0004,Customer 4,user4@example.com,2024-03-07,True,Mumbai,UK,True
4,C0005,Customer 5,user5@example.com,2025-03-23,False,Chennai,USA,True
...,...,...,...,...,...,...,...,...
115,C0116,Customer 116,user116@example.com,2024-12-09,False,London,UK,True
116,C0117,Customer 117,user117@example.com,2024-09-07,True,Chennai,Germany,True
117,C0118,Customer 118,user118@example.com,2025-01-08,True,Berlin,USA,True
118,C0119,Customer 119,user119@example.com,2025-04-23,True,Berlin,USA,True


## 10) Enrich currency, price -> Join exchange_rates ; amount_usd = amount × rate_to_usd (Reject if currency is UnKnown)

### Method - 1 :
#### using reset index -> changing column name -> merge the left and right table based on common column

In [54]:
exrates_df

,base,as_of,rates
USD,USD,2026-06-30,1.000
EUR,USD,2026-06-30,1.080
GBP,USD,2026-06-30,1.270
INR,USD,2026-06-30,0.012


In [55]:

new_rates = exrates_df.reset_index()
new_rates.rename(columns={"index": "currency"}, inplace=True)

In [56]:
new_rates = pd.DataFrame(new_rates,columns=["currency","rates"])
new_rates

,currency,rates
0,USD,1.000
1,EUR,1.080
2,GBP,1.270
3,INR,0.012


In [57]:
merged_df = pd.merge(
    left = ord_df,
    right = new_rates,
    on="currency",
    how = "left"

)

In [58]:
merged_df

,order_id,customer_id,product_id,quantity,amount,currency,order_date,rates
0,O00545,C0115,P0011,3,33.78,INR,2026-04-28,0.012
1,O00071,C0048,P0037,4,399.24,EUR,2026-03-21,1.080
2,O00550,C0103,P0059,4,494.92,EUR,2026-03-16,1.080
3,O00528,C0081,P0018,2,286.14,USD,2026-02-04,1.000
4,O00456,C0034,P0040,3,352.80,INR,2026-04-20,0.012
...,...,...,...,...,...,...,...,...
791,O00476,C0105,P0007,3,166.32,EUR,2026-02-02,1.080
792,O00813,C0113,P0053,4,1555.88,EUR,2026-02-01,1.080
793,O00360,C0037,P0051,5,235.80,INR,2026-02-21,0.012
794,O00128,C0082,P0099,3,1129.47,EUR,2026-06-30,1.080


### Method - 2 

In [59]:
exrates_df 

,base,as_of,rates
USD,USD,2026-06-30,1.000
EUR,USD,2026-06-30,1.080
GBP,USD,2026-06-30,1.270
INR,USD,2026-06-30,0.012


In [60]:
new_rates = pd.DataFrame(exrates_df,columns=["rates"])

In [61]:
new_rates

,rates
USD,1.000
EUR,1.080
GBP,1.270
INR,0.012


In [62]:
ord_df = pd.merge(
    ord_df,
    new_rates,
    how = "left",
    left_on = "currency",
    right_index=True
)


In [63]:
ord_df

,order_id,customer_id,product_id,quantity,amount,currency,order_date,rates
0,O00545,C0115,P0011,3,33.78,INR,2026-04-28,0.012
1,O00071,C0048,P0037,4,399.24,EUR,2026-03-21,1.080
2,O00550,C0103,P0059,4,494.92,EUR,2026-03-16,1.080
3,O00528,C0081,P0018,2,286.14,USD,2026-02-04,1.000
4,O00456,C0034,P0040,3,352.80,INR,2026-04-20,0.012
...,...,...,...,...,...,...,...,...
912,O00476,C0105,P0007,3,166.32,EUR,2026-02-02,1.080
913,O00813,C0113,P0053,4,1555.88,EUR,2026-02-01,1.080
914,O00360,C0037,P0051,5,235.80,INR,2026-02-21,0.012
915,O00128,C0082,P0099,3,1129.47,EUR,2026-06-30,1.080


In [64]:
ord_df["amount_usd"] = (ord_df["amount"] * ord_df["rates"]).round(2)

In [65]:
set(ord_df["currency"])

{'EUR', 'GBP', 'INR', 'JPY', 'USD'}

In [66]:
rejected_rates_rows = ord_df[ord_df["rates"].isnull()]
rejected_rates_rows

,order_id,customer_id,product_id,quantity,amount,currency,order_date,rates,amount_usd
916,O00481,C0069,P0040,5,588.0,JPY,2026-02-07,NaN,NaN


In [67]:
ord_df = ord_df.drop(index=rejected_rates_rows.index)

In [68]:
ord_df

,order_id,customer_id,product_id,quantity,amount,currency,order_date,rates,amount_usd
0,O00545,C0115,P0011,3,33.78,INR,2026-04-28,0.012,0.41
1,O00071,C0048,P0037,4,399.24,EUR,2026-03-21,1.080,431.18
2,O00550,C0103,P0059,4,494.92,EUR,2026-03-16,1.080,534.51
3,O00528,C0081,P0018,2,286.14,USD,2026-02-04,1.000,286.14
4,O00456,C0034,P0040,3,352.80,INR,2026-04-20,0.012,4.23
...,...,...,...,...,...,...,...,...,...
911,O00423,C0099,P0023,5,2203.80,INR,2026-03-30,0.012,26.45
912,O00476,C0105,P0007,3,166.32,EUR,2026-02-02,1.080,179.63
913,O00813,C0113,P0053,4,1555.88,EUR,2026-02-01,1.080,1680.35
914,O00360,C0037,P0051,5,235.80,INR,2026-02-21,0.012,2.83


## 11) Business Rule :
#### 1) total = qty×price ; 
#### 2) discount_value = total×discount/100 ;
#### 3) net_revenue = total − discount_value ;

In [69]:
discount = 10

ord_df["discount_value"] = (ord_df["amount"] * discount) / 100

ord_df["net_revenue"] = ord_df["amount"] - ord_df["discount_value"]

ord_df

,order_id,customer_id,product_id,quantity,amount,currency,order_date,rates,amount_usd,discount_value,net_revenue
0,O00545,C0115,P0011,3,33.78,INR,2026-04-28,0.012,0.41,3.378,30.402
1,O00071,C0048,P0037,4,399.24,EUR,2026-03-21,1.080,431.18,39.924,359.316
2,O00550,C0103,P0059,4,494.92,EUR,2026-03-16,1.080,534.51,49.492,445.428
3,O00528,C0081,P0018,2,286.14,USD,2026-02-04,1.000,286.14,28.614,257.526
4,O00456,C0034,P0040,3,352.80,INR,2026-04-20,0.012,4.23,35.280,317.520
...,...,...,...,...,...,...,...,...,...,...,...
911,O00423,C0099,P0023,5,2203.80,INR,2026-03-30,0.012,26.45,220.380,1983.420
912,O00476,C0105,P0007,3,166.32,EUR,2026-02-02,1.080,179.63,16.632,149.688
913,O00813,C0113,P0053,4,1555.88,EUR,2026-02-01,1.080,1680.35,155.588,1400.292
914,O00360,C0037,P0051,5,235.80,INR,2026-02-21,0.012,2.83,23.580,212.220


## 12) price_tier : low <10, medium 10–<100, high ≥ 100

In [70]:
product_df.head()

,product_id,product_name,category,unit_price,currency
0,P0001,Product 1,Books,384.61,INR
1,P0002,Product 2,Books,390.01,USD
2,P0003,Product 3,Electronics,58.51,INR
3,P0004,Product 4,Toys,290.02,INR
4,P0005,Product 5,Books,471.03,GBP


In [71]:

product_df["price_tier"] = "High"

product_df.loc[product_df["unit_price"]<10 , "price_tier"] = "Low"

product_df.loc[ (product_df["unit_price"]>=10) & (product_df["unit_price"]<100) , "price_tier"] = "Medium"

product_df

,product_id,product_name,category,unit_price,currency,price_tier
0,P0001,Product 1,Books,384.61,INR,High
1,P0002,Product 2,Books,390.01,USD,High
2,P0003,Product 3,Electronics,58.51,INR,Medium
3,P0004,Product 4,Toys,290.02,INR,High
4,P0005,Product 5,Books,471.03,GBP,High
5,P0006,Product 6,Grocery,298.09,USD,High
6,P0007,Product 7,Home,55.44,EUR,Medium
7,P0008,Product 8,Fashion,399.13,USD,High
8,P0009,Product 9,Toys,176.87,USD,High
9,P0010,Product 10,Grocery,39.13,INR,Medium


## 13) product_id of prodcuts must exist in product_id of orders

In [72]:
valid_product_id = set(product_df["product_id"])
valid_product_id

{'P0001',
 'P0002',
 'P0003',
 'P0004',
 'P0005',
 'P0006',
 'P0007',
 'P0008',
 'P0009',
 'P0010',
 'P0011',
 'P0012',
 'P0013',
 'P0014',
 'P0015',
 'P0016',
 'P0017',
 'P0018',
 'P0019',
 'P0020',
 'P0021',
 'P0022',
 'P0023',
 'P0024',
 'P0025',
 'P0026',
 'P0027',
 'P0028',
 'P0029',
 'P0030',
 'P0031',
 'P0032',
 'P0033',
 'P0034',
 'P0035',
 'P0036',
 'P0037',
 'P0038',
 'P0039',
 'P0040',
 'P0041',
 'P0042',
 'P0043',
 'P0044',
 'P0045',
 'P0046',
 'P0047',
 'P0048',
 'P0049',
 'P0050',
 'P0051',
 'P0052',
 'P0053',
 'P0054',
 'P0055',
 'P0056',
 'P0057',
 'P0058',
 'P0059',
 'P0060'}

In [73]:
invalid_product_id = ord_df[
   ~ord_df["product_id"].isin(valid_product_id)
   ]

In [74]:
invalid_product_id

,order_id,customer_id,product_id,quantity,amount,currency,order_date,rates,amount_usd,discount_value,net_revenue
915,O00128,C0082,P0099,3,1129.47,EUR,2026-06-30,1.08,1219.83,112.947,1016.523


In [75]:
ord_df =ord_df.drop(index = invalid_product_id.index)

In [76]:
max(ord_df["product_id"])

'P0060'

## 14) order_id vs returns
#### Left-join returns → add is_returned flag (keep non-returned)

In [77]:
ord_df

,order_id,customer_id,product_id,quantity,amount,currency,order_date,rates,amount_usd,discount_value,net_revenue
0,O00545,C0115,P0011,3,33.78,INR,2026-04-28,0.012,0.41,3.378,30.402
1,O00071,C0048,P0037,4,399.24,EUR,2026-03-21,1.080,431.18,39.924,359.316
2,O00550,C0103,P0059,4,494.92,EUR,2026-03-16,1.080,534.51,49.492,445.428
3,O00528,C0081,P0018,2,286.14,USD,2026-02-04,1.000,286.14,28.614,257.526
4,O00456,C0034,P0040,3,352.80,INR,2026-04-20,0.012,4.23,35.280,317.520
...,...,...,...,...,...,...,...,...,...,...,...
910,O00841,C0058,P0008,4,1596.52,USD,2026-03-06,1.000,1596.52,159.652,1436.868
911,O00423,C0099,P0023,5,2203.80,INR,2026-03-30,0.012,26.45,220.380,1983.420
912,O00476,C0105,P0007,3,166.32,EUR,2026-02-02,1.080,179.63,16.632,149.688
913,O00813,C0113,P0053,4,1555.88,EUR,2026-02-01,1.080,1680.35,155.588,1400.292


In [78]:
returns_df

,return_id,order_id,reason,return_date
0,R0001,O00286,NaN,2026-06-01
1,R0002,O00615,Wrong item,2026-07-08
2,R0003,O00353,Changed mind,2026-07-14
3,R0004,O00502,Wrong item,2026-03-18
4,R0005,O00383,Late,2026-04-09
...,...,...,...,...
124,R0125,O00121,Changed mind,2026-08-08
125,R0126,O00016,Damaged,2026-06-19
126,R0127,O00012,NaN,2026-03-29
127,R0128,O00059,NaN,2026-05-14


In [79]:
cust_df

,customer_id,name,email,signup_date,is_premium,city,country,email_present
0,C0001,Customer 1,user1@example.com,2025-12-07,True,London,India,True
1,C0002,Customer 2,user2@example.com,2025-09-19,False,Mumbai,USA,True
2,C0003,Customer 3,None,2025-09-28,True,Berlin,India,False
3,C0004,Customer 4,user4@example.com,2024-03-07,True,Mumbai,UK,True
4,C0005,Customer 5,user5@example.com,2025-03-23,False,Chennai,USA,True
...,...,...,...,...,...,...,...,...
115,C0116,Customer 116,user116@example.com,2024-12-09,False,London,UK,True
116,C0117,Customer 117,user117@example.com,2024-09-07,True,Chennai,Germany,True
117,C0118,Customer 118,user118@example.com,2025-01-08,True,Berlin,USA,True
118,C0119,Customer 119,user119@example.com,2025-04-23,True,Berlin,USA,True


In [80]:
ord_df["is_returned"] = ord_df["order_id"].isin(returns_df["order_id"])

## 15 Aggregate per customer_id
#### -> total_orders, total_spend_usd,
#### -> avg_order_value, return_rate, days_since_last_order

In [81]:
# shows duplicates customer_id rows in sorted order
duplicates  = ord_df[ord_df.duplicated(subset=["customer_id"],keep=False)].sort_values("customer_id")
duplicates

,order_id,customer_id,product_id,quantity,amount,currency,order_date,rates,amount_usd,discount_value,net_revenue,is_returned
50,O00826,C0001,P0043,3,1050.66,INR,2026-04-16,0.012,12.61,105.066,945.594,False
254,O00687,C0001,P0059,4,494.92,EUR,2026-06-08,1.080,534.51,49.492,445.428,False
524,O00695,C0001,P0053,4,1555.88,EUR,2026-06-30,1.080,1680.35,155.588,1400.292,False
24,O00195,C0001,P0055,1,156.16,GBP,2026-04-27,1.270,198.32,15.616,140.544,False
17,O00772,C0001,P0050,4,320.40,USD,2026-01-02,1.000,320.40,32.040,288.360,False
...,...,...,...,...,...,...,...,...,...,...,...,...
393,O00819,C0120,P0031,4,957.04,INR,2026-01-03,0.012,11.48,95.704,861.336,True
273,O00179,C0120,P0016,5,1175.55,GBP,2026-06-15,1.270,1492.95,117.555,1057.995,False
521,O00556,C0120,P0049,4,166.68,GBP,2026-05-02,1.270,211.68,16.668,150.012,False
133,O00377,C0120,P0023,1,440.76,INR,2026-04-03,0.012,5.29,44.076,396.684,False


In [82]:
cust_df.head()

,customer_id,name,email,signup_date,is_premium,city,country,email_present
0,C0001,Customer 1,user1@example.com,2025-12-07,True,London,India,True
1,C0002,Customer 2,user2@example.com,2025-09-19,False,Mumbai,USA,True
2,C0003,Customer 3,None,2025-09-28,True,Berlin,India,False
3,C0004,Customer 4,user4@example.com,2024-03-07,True,Mumbai,UK,True
4,C0005,Customer 5,user5@example.com,2025-03-23,False,Chennai,USA,True


In [83]:
product_df.head()

,product_id,product_name,category,unit_price,currency,price_tier
0,P0001,Product 1,Books,384.61,INR,High
1,P0002,Product 2,Books,390.01,USD,High
2,P0003,Product 3,Electronics,58.51,INR,Medium
3,P0004,Product 4,Toys,290.02,INR,High
4,P0005,Product 5,Books,471.03,GBP,High


In [84]:
returns_df.head()

,return_id,order_id,reason,return_date
0,R0001,O00286,NaN,2026-06-01
1,R0002,O00615,Wrong item,2026-07-08
2,R0003,O00353,Changed mind,2026-07-14
3,R0004,O00502,Wrong item,2026-03-18
4,R0005,O00383,Late,2026-04-09


In [85]:
ord_df.groupby("customer_id")

In [86]:
ord_df.groupby("customer_id")[["amount_usd"]].sum()

,amount_usd
customer_id,
C0001,4734.63
C0002,9288.09
C0003,5021.36
C0004,2704.77
C0005,2199.94
...,...
C0116,2325.81
C0117,4916.05
C0118,2129.00


In [87]:
type(
    ord_df.groupby("customer_id")["order_id"].count()
)

pandas.core.series.Series

In [88]:
type(
    ord_df.groupby("customer_id")[["order_id"]].count()
)

pandas.core.frame.DataFrame

In [89]:
ord_df.head()

,order_id,customer_id,product_id,quantity,amount,currency,order_date,rates,amount_usd,discount_value,net_revenue,is_returned
0,O00545,C0115,P0011,3,33.78,INR,2026-04-28,0.012,0.41,3.378,30.402,False
1,O00071,C0048,P0037,4,399.24,EUR,2026-03-21,1.080,431.18,39.924,359.316,False
2,O00550,C0103,P0059,4,494.92,EUR,2026-03-16,1.080,534.51,49.492,445.428,False
3,O00528,C0081,P0018,2,286.14,USD,2026-02-04,1.000,286.14,28.614,257.526,False
4,O00456,C0034,P0040,3,352.80,INR,2026-04-20,0.012,4.23,35.280,317.520,False


## Aggregation

In [90]:
aggregation =   ord_df.groupby("customer_id").agg({
                                                        "order_id" : "count",
                                                        "amount_usd" : ["sum","mean"]
                                                        })
                                        

In [91]:
aggregation

order_id amount_usd             
               count        sum         mean
customer_id                                 
C0001              6    4734.63   789.105000
C0002              8    9288.09  1161.011250
C0003              8    5021.36   627.670000
C0004              7    2704.77   386.395714
C0005              4    2199.94   549.985000
...              ...        ...          ...
C0116              4    2325.81   581.452500
C0117              8    4916.05   614.506250
C0118              5    2129.00   425.800000
C0119              9    5628.20   625.355556
C0120              7    2900.27   414.324286

[120 rows x 3 columns]

## Named Aggregation 

In [92]:
ord_df.head(2)

,order_id,customer_id,product_id,quantity,amount,currency,order_date,rates,amount_usd,discount_value,net_revenue,is_returned
0,O00545,C0115,P0011,3,33.78,INR,2026-04-28,0.012,0.41,3.378,30.402,False
1,O00071,C0048,P0037,4,399.24,EUR,2026-03-21,1.080,431.18,39.924,359.316,False


In [93]:
latest_order  = max(ord_df["order_date"])
print(latest_order.date())
type(latest_order)

2026-06-30


pandas._libs.tslibs.timestamps.Timestamp

In [94]:

import datetime as dt

today_date = dt.datetime.now().date()

print(type(today_date))

today_date = pd.to_datetime(today_date)

print(type(today_date))

<class 'datetime.date'>
<class 'pandas._libs.tslibs.timestamps.Timestamp'>


In [95]:
type(pd.Timestamp.now())

pandas._libs.tslibs.timestamps.Timestamp

In [96]:

named_aggregation = ord_df.groupby("customer_id").agg(
    total_orders = ("order_id","count"),
    total_spend_usd = ("amount_usd","sum"),
    avg_order_value = ("amount_usd","mean"),
    return_rate = ("is_returned","mean"),
    max_order_date = ("order_date","max")
)

In [97]:
named_aggregation["days_since_last_order"]  = today_date - (named_aggregation["max_order_date"])
named_aggregation

,total_orders,total_spend_usd,avg_order_value,return_rate,max_order_date,days_since_last_order
customer_id,,,,,,
C0001,6,4734.63,789.105000,0.166667,2026-06-30,21 days
C0002,8,9288.09,1161.011250,0.625000,2026-06-26,25 days
C0003,8,5021.36,627.670000,0.000000,2026-06-29,22 days
C0004,7,2704.77,386.395714,0.285714,2026-06-23,28 days
C0005,4,2199.94,549.985000,0.000000,2026-05-19,63 days
...,...,...,...,...,...,...
C0116,4,2325.81,581.452500,0.000000,2026-06-03,48 days
C0117,8,4916.05,614.506250,0.125000,2026-06-04,47 days
C0118,5,2129.00,425.800000,0.200000,2026-06-08,43 days


In [98]:
named_aggregation["days_since_last_order"] = (
    named_aggregation["days_since_last_order"].dt.days
)


In [99]:
named_aggregation

,total_orders,total_spend_usd,avg_order_value,return_rate,max_order_date,days_since_last_order
customer_id,,,,,,
C0001,6,4734.63,789.105000,0.166667,2026-06-30,21
C0002,8,9288.09,1161.011250,0.625000,2026-06-26,25
C0003,8,5021.36,627.670000,0.000000,2026-06-29,22
C0004,7,2704.77,386.395714,0.285714,2026-06-23,28
C0005,4,2199.94,549.985000,0.000000,2026-05-19,63
...,...,...,...,...,...,...
C0116,4,2325.81,581.452500,0.000000,2026-06-03,48
C0117,8,4916.05,614.506250,0.125000,2026-06-04,47
C0118,5,2129.00,425.800000,0.200000,2026-06-08,43


In [100]:
named_aggregation["return_rate"] = (
    (named_aggregation["return_rate"]*100).round(2).astype(str) + "%"
)

In [101]:
named_aggregation

,total_orders,total_spend_usd,avg_order_value,return_rate,max_order_date,days_since_last_order
customer_id,,,,,,
C0001,6,4734.63,789.105000,16.67%,2026-06-30,21
C0002,8,9288.09,1161.011250,62.5%,2026-06-26,25
C0003,8,5021.36,627.670000,0.0%,2026-06-29,22
C0004,7,2704.77,386.395714,28.57%,2026-06-23,28
C0005,4,2199.94,549.985000,0.0%,2026-05-19,63
...,...,...,...,...,...,...
C0116,4,2325.81,581.452500,0.0%,2026-06-03,48
C0117,8,4916.05,614.506250,12.5%,2026-06-04,47
C0118,5,2129.00,425.800000,20.0%,2026-06-08,43


## 16) Aggregate per customer_id (web_events) 
#### -> sessions_count,cart_abandon_rate

## Extraction : 

In [102]:
web_df.columns=[
    "timestamp",
    "session",
    "customer_id",
    "event",
    "product_id"
]
web_df

,timestamp,session,customer_id,event,product_id
0,2026-05-26T18:53:34,s5261,C0021,checkout,P0030
1,2026-03-02T04:48:04,s1248,C0036,remove_from_cart,P0006
2,2026-03-05T19:45:46,s5846,C0044,purchase,P0026
3,2026-01-10T22:24:41,s5493,C0047,remove_from_cart,P0043
4,2026-03-06T23:46:34,s1202,C0016,remove_from_cart,P0037
...,...,...,...,...,...
3995,2026-02-11T04:50:53,s1786,C0042,purchase,P0054
3996,2026-05-21T07:01:50,s8471,C0083,checkout,P0041
3997,2026-03-19T08:33:23,s6731,C0118,remove_from_cart,P0048
3998,2026-01-15T11:08:36,s9141,C0052,add_to_cart,P0022


#### Trying to get the values alone from rows 

In [103]:
stg ="session=s5261"

In [104]:
stg.strip().split("=")[1]

's5261'

In [105]:
web_df

,timestamp,session,customer_id,event,product_id
0,2026-05-26T18:53:34,s5261,C0021,checkout,P0030
1,2026-03-02T04:48:04,s1248,C0036,remove_from_cart,P0006
2,2026-03-05T19:45:46,s5846,C0044,purchase,P0026
3,2026-01-10T22:24:41,s5493,C0047,remove_from_cart,P0043
4,2026-03-06T23:46:34,s1202,C0016,remove_from_cart,P0037
...,...,...,...,...,...
3995,2026-02-11T04:50:53,s1786,C0042,purchase,P0054
3996,2026-05-21T07:01:50,s8471,C0083,checkout,P0041
3997,2026-03-19T08:33:23,s6731,C0118,remove_from_cart,P0048
3998,2026-01-15T11:08:36,s9141,C0052,add_to_cart,P0022


## Transform :

In [106]:
web_df["session"].sort_values()

666     s1000
2484    s1001
665     s1004
696     s1009
374     s1011
        ...  
2651    s9986
2773    s9986
2664    s9988
3805    s9988
389     s9993
Name: session, Length: 4000, dtype: object

In [107]:
session  = web_df.groupby("session")["customer_id"].nunique()

session

session
s1000    1
s1001    1
s1004    1
s1009    1
s1011    1
        ..
s9978    2
s9981    1
s9986    2
s9988    2
s9993    1
Name: customer_id, Length: 3221, dtype: int64

In [108]:
session[session > 1]

session
s1019    2
s1070    4
s1102    3
s1106    2
s1130    2
        ..
s9947    2
s9977    3
s9978    2
s9986    2
s9988    2
Name: customer_id, Length: 664, dtype: int64

In [109]:
web_df[web_df["session"]=="s1070"]

,timestamp,session,customer_id,event,product_id
895,2026-04-12T09:39:42,s1070,C0068,add_to_cart,P0003
2856,2026-03-19T16:53:28,s1070,C0021,purchase,P0036
3037,2026-06-04T07:34:25,s1070,C0105,add_to_cart,P0001
3987,2026-05-29T16:54:17,s1070,C0004,purchase,P0019


In [110]:
web_df.sort_values(by = "customer_id")

,timestamp,session,customer_id,event,product_id
90,2026-04-16T07:51:01,s5074,-,view,P0002
1003,2026-05-14T20:12:24,s9697,-,purchase,P0058
2518,2026-01-02T17:19:56,s8821,-,remove_from_cart,P0024
967,2026-05-05T14:00:07,s4586,-,purchase,P0033
2658,2026-04-03T22:11:49,s6985,-,view,P0005
...,...,...,...,...,...
350,2026-04-19T09:52:10,s4817,C0120,add_to_cart,P0050
2459,2026-06-17T23:57:54,s8641,C0120,checkout,P0010
1363,2026-01-24T23:35:28,s2376,C0120,add_to_cart,P0033
3662,2026-02-25T18:54:39,s3166,C0120,checkout,P0051


In [111]:
web_df

,timestamp,session,customer_id,event,product_id
0,2026-05-26T18:53:34,s5261,C0021,checkout,P0030
1,2026-03-02T04:48:04,s1248,C0036,remove_from_cart,P0006
2,2026-03-05T19:45:46,s5846,C0044,purchase,P0026
3,2026-01-10T22:24:41,s5493,C0047,remove_from_cart,P0043
4,2026-03-06T23:46:34,s1202,C0016,remove_from_cart,P0037
...,...,...,...,...,...
3995,2026-02-11T04:50:53,s1786,C0042,purchase,P0054
3996,2026-05-21T07:01:50,s8471,C0083,checkout,P0041
3997,2026-03-19T08:33:23,s6731,C0118,remove_from_cart,P0048
3998,2026-01-15T11:08:36,s9141,C0052,add_to_cart,P0022


In [112]:
web_df[["session","customer_id"]].sort_values(by="customer_id",ascending=False)

,session,customer_id
528,s8805,C0120
3907,s4079,C0120
350,s4817,C0120
3662,s3166,C0120
1363,s2376,C0120
...,...,...
2372,s1549,-
3430,s4367,-
2228,s8835,-
1507,s7599,-


## Method - 1 without using pandas

In [113]:

grouped = web_df.groupby(["customer_id", "session"])
session_summary= []

for (customer_id, session), group in grouped:

    events = group["event"].tolist()

    has_cart = "add_to_cart" in events      
    has_purchase = "purchase" in events    
    abandoned = has_cart and not has_purchase

    session_summary.append({
        "customer_id" : customer_id ,
        "session" : session,
        "has_cart" : has_cart,
        "has_purchase" : has_purchase,
        "abandoned" : abandoned
    }) 
session_summary = pd.DataFrame(session_summary)
session_summary

,customer_id,session,has_cart,has_purchase,abandoned
0,-,s1159,False,False,False
1,-,s1232,True,False,True
2,-,s1259,False,False,False
3,-,s1295,False,False,False
4,-,s1392,False,False,False
...,...,...,...,...,...
3988,C0120,s8792,False,True,False
3989,C0120,s8805,False,True,False
3990,C0120,s9576,False,False,False
3991,C0120,s9605,False,False,False


In [114]:
session_summary = (
            web_df
            .groupby(["customer_id"])
            .agg(events = ("event", list))
        )

In [115]:
session_summary

,events
customer_id,
-,"[remove_from_cart, view, purchase, view, view,..."
C0001,"[checkout, add_to_cart, checkout, purchase, ad..."
C0002,"[checkout, checkout, remove_from_cart, view, a..."
C0003,"[purchase, checkout, checkout, purchase, add_t..."
C0004,"[add_to_cart, add_to_cart, add_to_cart, purcha..."
...,...
C0116,"[checkout, purchase, add_to_cart, checkout, pu..."
C0117,"[remove_from_cart, remove_from_cart, add_to_ca..."
C0118,"[purchase, checkout, view, add_to_cart, add_to..."


## Method - 2 using pandas

In [116]:
session_summary = (
    web_df
    .groupby(["customer_id", "session"])
    .agg(events=("event", set))
)

In [117]:
session_summary["has_cart"] = session_summary["events"].apply(lambda x: "add_to_cart" in x)

session_summary["abandoned"] = session_summary["events"].apply(lambda x: "remove_from_cart" in x)

    

In [118]:
session_summary

events  has_cart  abandoned
customer_id session                                         
-           s1159    {remove_from_cart}     False       True
            s1232         {add_to_cart}      True      False
            s1259    {remove_from_cart}     False       True
            s1295                {view}     False      False
            s1392            {checkout}     False      False
...                                 ...       ...        ...
C0120       s8792            {purchase}     False      False
            s8805            {purchase}     False      False
            s9576            {checkout}     False      False
            s9605    {remove_from_cart}     False       True
            s9894         {add_to_cart}      True      False

[3993 rows x 3 columns]

In [119]:
session_summary = session_summary.reset_index()

In [120]:
customer_web_features = (
    session_summary
    .groupby("customer_id")
    .agg (
        sessions_count = ("session", "nunique"),
        sessions_with_cart = ("has_cart" , "sum"),
        abandoned_sessions = ("abandoned", "sum")
     ) 
)

In [121]:
customer_web_features["abandoned_rate"] =  customer_web_features["abandoned_sessions"]/customer_web_features["sessions_with_cart"] 

In [122]:
customer_web_features

,sessions_count,sessions_with_cart,abandoned_sessions,abandoned_rate
customer_id,,,,
-,65,9,10,1.111111
C0001,43,7,9,1.285714
C0002,28,6,8,1.333333
C0003,31,8,2,0.250000
C0004,36,5,7,1.400000
...,...,...,...,...
C0116,26,7,1,0.142857
C0117,35,6,4,0.666667
C0118,39,9,6,0.666667


In [123]:

customer_feature_table = pd.merge(
    named_aggregation,
    customer_web_features,
    how = 'left',
    left_on = "customer_id",
    right_index= True
)

In [124]:
customer_feature_table

,total_orders,total_spend_usd,avg_order_value,return_rate,max_order_date,days_since_last_order,sessions_count,sessions_with_cart,abandoned_sessions,abandoned_rate
customer_id,,,,,,,,,,
C0001,6,4734.63,789.105000,16.67%,2026-06-30,21,43,7,9,1.285714
C0002,8,9288.09,1161.011250,62.5%,2026-06-26,25,28,6,8,1.333333
C0003,8,5021.36,627.670000,0.0%,2026-06-29,22,31,8,2,0.250000
C0004,7,2704.77,386.395714,28.57%,2026-06-23,28,36,5,7,1.400000
C0005,4,2199.94,549.985000,0.0%,2026-05-19,63,28,3,9,3.000000
...,...,...,...,...,...,...,...,...,...,...
C0116,4,2325.81,581.452500,0.0%,2026-06-03,48,26,7,1,0.142857
C0117,8,4916.05,614.506250,12.5%,2026-06-04,47,35,6,4,0.666667
C0118,5,2129.00,425.800000,20.0%,2026-06-08,43,39,9,6,0.666667


In [125]:
web_df

,timestamp,session,customer_id,event,product_id
0,2026-05-26T18:53:34,s5261,C0021,checkout,P0030
1,2026-03-02T04:48:04,s1248,C0036,remove_from_cart,P0006
2,2026-03-05T19:45:46,s5846,C0044,purchase,P0026
3,2026-01-10T22:24:41,s5493,C0047,remove_from_cart,P0043
4,2026-03-06T23:46:34,s1202,C0016,remove_from_cart,P0037
...,...,...,...,...,...
3995,2026-02-11T04:50:53,s1786,C0042,purchase,P0054
3996,2026-05-21T07:01:50,s8471,C0083,checkout,P0041
3997,2026-03-19T08:33:23,s6731,C0118,remove_from_cart,P0048
3998,2026-01-15T11:08:36,s9141,C0052,add_to_cart,P0022
